In [3]:
import os
import time
import pandas as pd
import yfinance as yf

In [4]:
nasdaq_df = pd.read_csv(
    r"C:\Users\Aamir Bin Abbas\Desktop\Research & Thesis\Quantitative Stock Market Analysis\data\raw\nasdaq_screener_1793773127340.csv"
)

In [5]:
nasdaq_df.groupby(
    ["Symbol", "Name", "Sector"],
    dropna=True,
).size().reset_index()

,Symbol,Name,Sector,0
0,A,Agilent Technologies Inc. Common Stock,Industrials,1
1,AA,Alcoa Corporation Common Stock,Industrials,1
2,AAC,Ares Acquisition Corporation III Class A Ordin...,Industrials,1
3,AACG,ATA Creativity Global American Depositary Shares,Real Estate,1
4,AACI,Armada Acquisition Corp. III Class A Ordinary ...,Finance,1
...,...,...,...,...
6318,ZVIA,Zevia PBC Class A Common Stock,Consumer Staples,1
6319,ZVRA,Zevra Therapeutics Inc. Common Stock,Health Care,1
6320,ZWS,Zurn Elkay Water Solutions Corporation Common ...,Industrials,1
6321,ZYBT,Zhengye Biotechnology Holding Limited Class A ...,Health Care,1


In [6]:
sector_type = set()

for st in nasdaq_df["Sector"]:
    if isinstance(st, str):
        sector_type.add(st.strip())

In [7]:
nasdaq_df.groupby("Sector").sample(n=1)

,Symbol,Name,Last Sale,Net Change,% Change,Market Cap,Country,IPO Year,Volume,Sector,Industry
4202,MUX,McEwen Inc. Common Stock,$14.05,0.02,0.143%,8.620879e+08,Canada,NaN,1282228,Basic Materials,Precious Metals
6194,TKLF,Tokyo Lifestyle Co. Ltd. American Depositary S...,$1.39,-0.11,-7.333%,7.760572e+06,Japan,2022.0,4337,Consumer Discretionary,Other Specialty Stores
3736,LMNR,Limoneira Co Common Stock,$10.64,-0.04,-0.375%,1.929246e+08,United States,NaN,165507,Consumer Staples,Farming/Seeds/Milling
2711,GLND,Greenland Energy Company Common Stock,$2.68,-0.41,-13.269%,1.171969e+08,United States,NaN,2062488,Energy,Oil & Gas Production
3575,KRNY,Kearny Financial Corp Common Stock,$9.36,0.24,2.632%,6.077390e+08,United States,2005.0,453602,Finance,Savings Institutions
4650,OMDA,Omada Health Inc. Common Stock,$17.85,-0.88,-4.698%,1.092718e+09,United States,2025.0,948461,Health Care,Medical/Nursing Services
3319,IR,Ingersoll Rand Inc. Common Stock,$82.73,2.68,3.348%,3.176996e+10,United States,2017.0,5563106,Industrials,Industrial Machinery/Components
6671,VYX,NCR Voyix Corporation Common Stock,$8.92,0.44,5.189%,1.231583e+09,United States,NaN,3667668,Miscellaneous,Office Equipment/Supplies/Services
2030,EEIQ,EpicQuest Education Group International Limite...,$4.12,0.00,0.00%,6.784087e+06,United States,2021.0,306,Real Estate,Other Consumer Services
3283,IONQ,IonQ Inc. Common Stock,$28.17,-1.87,-6.225%,1.151893e+10,United States,2021.0,20840758,Technology,EDP Services


In [ ]:
# this function downloads datasets sector wise for EDA. it names the file according to its name in original dataframe
def get_sector_files(period: str, interval: str) -> None | int:

    count = 0  # for keeping account of data collected

    try:
        # new folder for sector wise data
        full_path = r""  # can be updated when required

        # to make directory named 'sector_wise_csv'
        os.makedirs(full_path, exist_ok=True)

        # obtaining random sample 'Sector' wise to analyze data
        df = nasdaq_df.groupby("Sector").sample(n=1)

        # Process each selected company
        for _, row in df.iterrows():
            # Extract company info
            target_name = row["Name"]
            target_sector = row["Sector"]
            ticker = row["Symbol"]

            # validate ticker
            if not isinstance(ticker, str) or not ticker.strip():
                print(f"Invalid ticker for {target_sector}")
                continue

            # generate file names
            file_name = f"{target_name}_{target_sector}_data_file.csv".replace(" ", "_")

            # joining and storing the data files in full_path with respective names
            sector_path = os.path.join(full_path, file_name)

            time.sleep(2)

            data = yf.download(
                ticker, period=period, interval=interval, auto_adjust=True
            )

            if data is None or data.empty:
                print("Data is not available")
                continue

            # saving the obtained data to csv
            data.to_csv(sector_path, mode="x")
            print(f"Data file named {file_name} has been saved !")
            count += 1

        return count

    except Exception as e:
        print(f"Could not fetch data :  {e}")
        return None

In [10]:
# get_sector_files("5y", "1d")

# run this only when needed

In [11]:
# extracting ticker symbols from the nasdaq csv file (this is not historical dataset and only be used for reference and tickers)
ticker_list = []
for sym in nasdaq_df["Symbol"]:
    if isinstance(sym, str) and sym.strip():
        ticker_list.append(sym.strip())

In [12]:
# extracting ticker symbols from the nasdaq csv file (this is not historical dataset and only be used for reference and tickers)
name_list = []
for sym in nasdaq_df["Name"]:
    name_list.append(sym)

In [14]:
# below function extracts singular file data via yahoo finance and the data will be saved in raw data subfolder in data.
def get_data_file(
    ticker: str, period: str, interval: str, f_name: str
) -> pd.DataFrame | None:

    full_path = r"C:\Users\Aamir Bin Abbas\Desktop\Research & Thesis\Stock-Price-Predictor-Close--LSTM\data\raw\yahoo_finance"

    try:
        # create a file inside "full_path"
        os.makedirs(full_path, exist_ok=True)

        # naming structure of individual files
        file_name = f"{f_name}_stock_data.csv".replace(" ", "_")

        # csv file to be saved in raw subfolder in data folder
        raw_path = os.path.join(full_path, file_name)

        time.sleep(2)  # precaution to avoid rate limiter by yf.download
        data = yf.download(ticker, period=period, interval=interval, auto_adjust=True)

        if data is None or data.empty:
            print("No data was found in ")
            return None

        data.to_csv(
            raw_path, mode="x"
        )  # the mode can be changed after downloading to x if file already exits

        print(f"Data file named {file_name} has been saved !")

        return data

    except Exception:
        print(f"Error : Either the file exists or Unavailable")

In [ ]:
# run this when file is required to be downloaded : first 20 files shall be downloaded
for i in range(0, 20):
    target_ticker = ticker_list[i]
    target_filename = name_list[i]

    try:
        get_data_file(target_ticker, "5y", "1d", target_filename)
    except Exception as e:
        print(f"File Already Exists : {e}")